# TimeGAN Model

In [1]:
import pandas as pd
import numpy as np
from ydata_synthetic.synthesizers.timeseries import TimeSeriesSynthesizer
from ydata_synthetic.synthesizers import ModelParameters, TrainParameters
import os
import joblib

# Define Dataset configuration to handle different datasets
class DatasetConfig:
    def __init__(self, name, csv_path, window_size, output_dir):
        self.name = name
        self.csv_path = csv_path
        self.window_size = window_size
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

Define TimeGAN manager to handle different datasets with specific configuration.

In [2]:
class TimeGANManager:
    def __init__(self, config, scaler_path):
        self.config = config
        os.makedirs(self.config.output_dir, exist_ok=True)
        self.model_path = os.path.join(self.config.output_dir, f"{self.config.name}.pkl")
        
        self.df = pd.read_csv(self.config.csv_path)
        self.numeric_cols = self.df.columns.tolist()
        
        self.scaler = joblib.load(scaler_path)
        self.model = None

        

    def train(self, epochs=50000, batch_size=128):
        print(f"--- Training TimeGAN ({self.config.name}) ---")

        if os.path.exists(self.model_path):
            print("    > Loading existing TimeGAN model...")
            self.model = TimeSeriesSynthesizer.load(self.model_path)
            return

        model_args = ModelParameters(
            batch_size=batch_size,
            lr=2e-4,
            noise_dim=64,
            layers_dim=128,
            latent_dim=24,
            gamma=1.5
        )

        train_args = TrainParameters(
            epochs=epochs,
            sequence_length=self.config.window_size,
            number_sequences=len(self.numeric_cols)
        )

        self.model = TimeSeriesSynthesizer(
            modelname="timegan",
            model_parameters=model_args
        )

        self.model.fit(
            data=self.df, 
            train_arguments=train_args,
            num_cols=self.numeric_cols,
            cat_cols=[]
        )

        self.model.save(self.model_path)
        print("    > TimeGAN training complete.")

    

    def generate(self, num_samples=None):
        if num_samples == None:
            num_samples = len(self.df) // self.config.window_size

        synth_blocks = self.model.sample(n_samples=num_samples)
        synth_df = pd.concat(synth_blocks, axis=0).reset_index(drop=True)

        # Inverse scaling
        synth_df[self.numeric_cols] = self.scaler.inverse_transform(
            synth_df[self.numeric_cols]
        )


        output_path = self.config.csv_path.replace(
            "numerical.csv", "timegan_synthetic.csv"
        )

        synth_df.round(2).to_csv(output_path, index=False)
        print(f"    > Saved to {output_path}")

        # return synth_df

Generate synthetic data for different datasets with specific window size.

In [3]:
WINDOW_SIZE_DAILY = 30
WINDOW_SIZE_HOURLY = 48
WINDOW_SIZE_MINUTELY = 80

# Google Stocks Data
cfg_stocks = DatasetConfig("GOOGL_Stocks", "./data/GOOGL_STOCKS/GOOGL_STOCKS_numerical.csv", WINDOW_SIZE_DAILY, "./timegan_models")
gan_stocks = TimeGANManager(cfg_stocks, './scaler/GOOGL_STOCKS_scaler.pkl')
gan_stocks.train(epochs=600, batch_size=128)
gan_stocks.generate()

# Occupancy Detection Data
cfg_occupancy = DatasetConfig("OccupancyDetection", "./data/OccupancyDetection/OccupancyDetection_numerical.csv", WINDOW_SIZE_MINUTELY, "./timegan_models")
gan_occupancy = TimeGANManager(cfg_occupancy, './scaler/OccupancyDetection_scaler.pkl')
gan_occupancy.train(epochs=600, batch_size=128)
gan_occupancy.generate()

# Power Consumption Data
cfg_power = DatasetConfig("PowerConsumption", "./data/PowerConsumption/PowerConsumption_numerical.csv", WINDOW_SIZE_MINUTELY, "./timegan_models")
gan_power = TimeGANManager(cfg_power, './scaler/PowerConsumption_scaler.pkl')
gan_power.train(epochs=1000, batch_size=64)
gan_power.generate()

# Traffic Volume Data
cfg_traffic = DatasetConfig("TrafficVolume", "./data/TrafficVolume/TrafficVolume_numerical.csv", WINDOW_SIZE_HOURLY, "./timegan_models")
gan_traffic = TimeGANManager(cfg_traffic, './scaler/TrafficVolume_scaler.pkl')
gan_traffic.train(epochs=600, batch_size=128)
gan_traffic.generate()

--- Training TimeGAN (GOOGL_Stocks) ---
    > Loading existing TimeGAN model...
    > Saved to ./data/GOOGL_STOCKS/GOOGL_STOCKS_timegan_synthetic.csv
--- Training TimeGAN (OccupancyDetection) ---
    > Loading existing TimeGAN model...
    > Saved to ./data/OccupancyDetection/OccupancyDetection_timegan_synthetic.csv
--- Training TimeGAN (PowerConsumption) ---
    > Loading existing TimeGAN model...
    > Saved to ./data/PowerConsumption/PowerConsumption_timegan_synthetic.csv
--- Training TimeGAN (TrafficVolume) ---
    > Loading existing TimeGAN model...
    > Saved to ./data/TrafficVolume/TrafficVolume_timegan_synthetic.csv
